In [2]:
import os
import sys

project_root = r"C:\Users\User\pg-screening-system"
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Root :", os.getcwd())

Root : C:\Users\User\pg-screening-system


In [3]:
import subprocess
result = subprocess.run(
    ["python", "-c",
     "import src.extractors.university_extractor as m; print(dir(m))"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "C:\Users\User\pg-screening-system\src\extractors\university_extractor.py", line 5, in <module>
    from rapidfuzz import process, fuzz
ModuleNotFoundError: No module named 'rapidfuzz'



In [16]:
append_code = '''

def split_bsc_msc(education_text: str) -> dict:
    """Sépare le texte EDUCATION en parties BSc et MSc."""
    import re
    bsc_keywords = r\'(?i)(bachelor|b\\.sc|bsc|b\\.s\\.|undergraduate|licence)\'
    msc_keywords = r\'(?i)(master|m\\.sc|msc|m\\.s\\.|postgraduate)\'
    lines = education_text.split(\'\\n\')
    bsc_lines, msc_lines = [], []
    current = None
    for line in lines:
        if re.search(msc_keywords, line):
            current = \'msc\'
        elif re.search(bsc_keywords, line):
            current = \'bsc\'
        if current == \'bsc\':
            bsc_lines.append(line)
        elif current == \'msc\':
            msc_lines.append(line)
    return {"bsc": \'\\n\'.join(bsc_lines), "msc": \'\\n\'.join(msc_lines)}
'''

with open("src/extractors/university_extractor.py", "a", encoding="utf-8") as f:
    f.write(append_code)

print("Fonction ajoutée.")

# Vérification immédiate
with open("src/extractors/university_extractor.py", "r", encoding="utf-8") as f:
    content = f.read()

if "def split_bsc_msc" in content:
    print("split_bsc_msc présente dans le fichier ✅")
else:
    print("ABSENT ❌")

Fonction ajoutée.
split_bsc_msc présente dans le fichier ✅


In [6]:
import os, sys
project_root = r"C:\Users\User\pg-screening-system"
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("Root :", os.getcwd())

Root : C:\Users\User\pg-screening-system


In [7]:
import json, re, fitz
import pandas as pd
from src.preprocessing.segmenter import segment_document
from src.extractors.gpa_extractor import extract_gpa
from src.extractors.university_extractor import (
    extract_university, load_qs_db, split_bsc_msc)
from src.extractors.publication_extractor import (
    extract_publications, load_reference_data)
print("Imports OK ✅")

Imports OK ✅


In [8]:
df_qs, qs_names    = load_qs_db("data/qs_rankings_2025.csv")
df_scopus, df_core = load_reference_data("data/scopus_sjr.csv",
                                          "data/core_rankings.csv")
doc     = fitz.open("data/raw/cvs/257264722 CV.pdf")
text_cv = "".join(p.get_text() for p in doc)
seg     = segment_document(text_cv)
print("Sections :", seg.detected_section_names)
print("Confiance :", seg.confidence)

Sections : ['HEADER', 'SUMMARY', 'SKILLS', 'EDUCATION', 'EXPERIENCE', 'PUBLICATIONS', 'PROJECTS']
Confiance : 1.0


In [9]:
# Extraire toutes les métriques NLP
edu_text = seg.sections.get("EDUCATION", "")
pub_text = seg.sections.get("PUBLICATIONS", "")

# GPA
parts    = split_bsc_msc(edu_text)
gpa_bsc  = extract_gpa(parts['bsc'], degree_level="bsc")
gpa_msc  = extract_gpa(parts['msc'], degree_level="msc")

# Universités
univ_bsc = extract_university(parts['bsc'], df_qs, qs_names, "bsc")
univ_msc = extract_university(parts['msc'], df_qs, qs_names, "msc")

# Publications
pubs = extract_publications(pub_text, df_scopus, df_core,
                             candidate_hint="Al-Nashash")

print("=== RÉSULTATS NLP ===")
print(f"BSc GPA    : {gpa_bsc.normalised_gpa} (conf={gpa_bsc.confidence})")
print(f"MSc GPA    : {gpa_msc.normalised_gpa} (conf={gpa_msc.confidence})")
print(f"BSc Univ   : {univ_bsc.detected_name} | QS={univ_bsc.qs_rank} (conf={univ_bsc.confidence})")
print(f"MSc Univ   : {univ_msc.detected_name} | QS={univ_msc.qs_rank} (conf={univ_msc.confidence})")
print(f"Pubs       : {len(pubs)} trouvées")
for p in pubs:
    print(f"  → {p.title[:50]}... | conf={p.confidence}")

=== RÉSULTATS NLP ===
BSc GPA    : None (conf=0.0)
MSc GPA    : None (conf=0.0)
BSc Univ   : AlHosn University | QS=None (conf=0.1)
MSc Univ   : New York University | QS=32 (conf=1.0)
Pubs       : 1 trouvées
  → Enhancing IoT Security using Blockchain-Based
Acce... | conf=0.59


In [10]:
from dataclasses import dataclass, field
from typing import Optional, List

CONFIDENCE_THRESHOLD = 0.70   # seuil configurable

@dataclass
class FieldConfidence:
    field_name:  str
    value:       object
    confidence:  float
    needs_llm:   bool
    source:      str          # nlp | llm | manual

@dataclass
class GlobalConfidenceReport:
    fields:          List[FieldConfidence]
    global_score:    float
    fields_ok:       List[str]
    fields_llm:      List[str]
    trigger_llm:     bool

def compute_confidence_report(
        gpa_bsc, gpa_msc,
        univ_bsc, univ_msc,
        pubs,
        threshold: float = CONFIDENCE_THRESHOLD
) -> GlobalConfidenceReport:

    fields = [
        FieldConfidence("bsc_gpa",
                        gpa_bsc.normalised_gpa,
                        gpa_bsc.confidence,
                        gpa_bsc.confidence < threshold,
                        "nlp"),
        FieldConfidence("msc_gpa",
                        gpa_msc.normalised_gpa,
                        gpa_msc.confidence,
                        gpa_msc.confidence < threshold,
                        "nlp"),
        FieldConfidence("bsc_university",
                        univ_bsc.detected_name,
                        univ_bsc.confidence,
                        univ_bsc.confidence < threshold,
                        "nlp"),
        FieldConfidence("msc_university",
                        univ_msc.detected_name,
                        univ_msc.confidence,
                        univ_msc.confidence < threshold,
                        "nlp"),
    ]

    # Ajouter les publications
    for i, pub in enumerate(pubs[:3]):
        fields.append(FieldConfidence(
            f"publication_{i+1}",
            pub.title,
            pub.confidence,
            pub.confidence < threshold,
            "nlp"
        ))

    global_score = round(
        sum(f.confidence for f in fields) / len(fields), 3)

    fields_ok  = [f.field_name for f in fields if not f.needs_llm]
    fields_llm = [f.field_name for f in fields if f.needs_llm]

    return GlobalConfidenceReport(
        fields=fields,
        global_score=global_score,
        fields_ok=fields_ok,
        fields_llm=fields_llm,
        trigger_llm=len(fields_llm) > 0
    )

report = compute_confidence_report(
    gpa_bsc, gpa_msc, univ_bsc, univ_msc, pubs)

print("=== CONFIDENCE REPORT ===")
print(f"Score global : {report.global_score}")
print(f"Seuil        : {CONFIDENCE_THRESHOLD}")
print()
print(f"Champs OK ({len(report.fields_ok)}) :")
for f in report.fields_ok:
    print(f"  ✅ {f}")
print()
print(f"Champs → LLM ({len(report.fields_llm)}) :")
for f in report.fields_llm:
    print(f"  ⚠️  {f}")
print()
print(f"Déclencher LLM : {report.trigger_llm}")

=== CONFIDENCE REPORT ===
Score global : 0.338
Seuil        : 0.7

Champs OK (1) :
  ✅ msc_university

Champs → LLM (4) :
  ⚠️  bsc_gpa
  ⚠️  msc_gpa
  ⚠️  bsc_university
  ⚠️  publication_1

Déclencher LLM : True


In [13]:
#Cellule diagnostic — tester Ollama
import requests, json

# Test simple
response = requests.post("http://localhost:11434/api/generate", json={
    "model": "mistral:7b-instruct",
    "prompt": 'Return this JSON exactly: {"test": "ok"}',
    "stream": False,
    "options": {"temperature": 0.1}
}, timeout=60)

print("Status :", response.status_code)
raw = response.json().get("response", "")
print("Réponse brute :")
print(repr(raw[:500]))

Status : 200
Réponse brute :
' Here is the requested JSON:\n\n```json\n{"test": "ok"}\n```'


In [15]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral:7b-instruct"  # ← nom exact vu dans la liste

def call_ollama_v2(section_text: str, fields_needed: list) -> dict:
    schema = {f: None for f in fields_needed}

    prompt = (
        "You are an academic CV parser. "
        "Extract ONLY what is explicitly written. "
        "Return ONLY valid JSON, no explanation.\n\n"
        f"Fields needed: {fields_needed}\n\n"
        f"Return format: {json.dumps(schema)}\n\n"
        f"Text:\n{section_text[:1200]}\n\n"
        "JSON:"
    )

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.1}
            },
            timeout=120
        )

        if response.status_code != 200:
            print(f"Erreur HTTP : {response.status_code}")
            return {}

        raw = response.json().get("response", "").strip()
        print("Réponse brute LLM :")
        print(raw[:600])

        # Extraire JSON
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
        return {}

    except Exception as e:
        print(f"Erreur : {e}")
        return {}

# Test sur les champs manquants d'Esra
section_for_llm = (seg.sections.get("EDUCATION", "")
                   + "\n"
                   + seg.sections.get("PUBLICATIONS", ""))

print("Texte envoyé au LLM :")
print(section_for_llm[:400])
print("\n--- Appel LLM ---")

llm_result = call_ollama_v2(section_for_llm, report.fields_llm)

print("\nRésultat parsé :")
print(json.dumps(llm_result, indent=2))

Texte envoyé au LLM :
Master of Science in Cybersecurity - Summer 2024
New York University, New York

Bachelor of Science in Software Engineering - Spring 2017
AlHosn University, Abu Dhabi, UAE
 Al-Nashash, E. J., & Olmsted, A. (2023). Enhancing IoT Security using Blockchain-Based
Access Control Mechanisms and Database Management. In Proceedings of the
International Conference for Internet Technology and Secured Trans

--- Appel LLM ---
Réponse brute LLM :
{"bsc_gpa": null, "msc_gpa": null, "bsc_university": "AlHosn University, Abu Dhabi, UAE", "publication_1": "Al-Nashash, E. J., & Olmsted, A. (2023). Enhancing IoT Security using Blockchain-Based Access Control Mechanisms and Database Management. In Proceedings of the International Conference for Internet Technology and Secured Transactions (ICITST-2023), St Anne’s College, Oxford University, UK, 13-15 November 2023. DOI: 10.20533/ICITST.2023.0021. ISBN: 978-1-913572-63-1."}

Résultat parsé :
{
  "bsc_gpa": null,
  "msc_gpa": null,
 

In [16]:
def fuse_results(report: GlobalConfidenceReport,
                 llm_result: dict) -> dict:
    """
    Règles de fusion :
    1. NLP conf ≥ 0.70 → garder NLP
    2. NLP conf < 0.70 ET LLM disponible → prendre LLM
    3. Les deux échouent → NULL + needs_review
    """
    final = {}

    for f in report.fields:
        if not f.needs_llm:
            # NLP suffisant
            final[f.field_name] = {
                "value":  f.value,
                "source": "nlp",
                "confidence": f.confidence
            }
        elif f.field_name in llm_result and llm_result[f.field_name]:
            # LLM a trouvé quelque chose
            final[f.field_name] = {
                "value":  llm_result[f.field_name],
                "source": "llm",
                "confidence": 0.65   # confiance LLM par défaut
            }
        else:
            # Les deux ont échoué
            final[f.field_name] = {
                "value":  None,
                "source": "failed",
                "confidence": 0.0,
                "needs_review": True
            }

    return final

fused = fuse_results(report, llm_result)

print("=== RÉSULTAT FINAL FUSIONNÉ ===")
for field, data in fused.items():
    icon = "✅" if data['source'] == "nlp" else \
           "🤖" if data['source'] == "llm" else "❌"
    print(f"{icon} {field:<20} : {data['value']} "
          f"(source={data['source']}, conf={data['confidence']})")

=== RÉSULTAT FINAL FUSIONNÉ ===
❌ bsc_gpa              : None (source=failed, conf=0.0)
❌ msc_gpa              : None (source=failed, conf=0.0)
🤖 bsc_university       : AlHosn University, Abu Dhabi, UAE (source=llm, conf=0.65)
✅ msc_university       : New York University (source=nlp, conf=1.0)
🤖 publication_1        : Al-Nashash, E. J., & Olmsted, A. (2023). Enhancing IoT Security using Blockchain-Based Access Control Mechanisms and Database Management. In Proceedings of the International Conference for Internet Technology and Secured Transactions (ICITST-2023), St Anne’s College, Oxford University, UK, 13-15 November 2023. DOI: 10.20533/ICITST.2023.0021. ISBN: 978-1-913572-63-1. (source=llm, conf=0.65)


In [9]:
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = r.json().get("models", [])
    print("Ollama disponible ✅")
    print("Modèles :", [m['name'] for m in models])
except:
    print("Ollama non disponible ⚠️")
    print("Lancez : docker compose up -d ollama")
    print("Puis   : docker exec pgars-ollama ollama pull mistral")


Ollama disponible ✅
Modèles : ['mistral:7b-instruct']
